# RTVIO batch reconstruction on Kaggle (free T4)

Runs the **offline/batch** VGGT reconstruction path (`rtvio.vggt_reconstruct`) on a Kaggle GPU notebook, using a free T4 (16GB VRAM) instead of a low-VRAM local laptop GPU.

**What this does *not* do**: it does not run the live `RTVIO Studio` server that a phone/drone streams into in real time (`phone_link.py` / `drone_link.py`). Kaggle notebooks have no reachable public inbound network address, so there is nothing for the phone/drone to connect to here. Workflow is instead:

1. Capture/record a session locally (phone app or drone), or have a plain video file + optional GPS CSV.
2. Upload that recording (session folder or video file) to Kaggle as a private Dataset, or drag it into this notebook's "Data" pane.
3. Run reconstruction here on the T4.
4. Download the resulting point cloud / mesh / report from `/kaggle/working`.

**Hardware note**: T4 is a Turing-generation GPU (compute capability 7.5), same tier as the project's originally-tested "earlier GPU" (GTX 1650). `_load_vggt()` already handles this: below Ampere (`sm_80`) it runs the aggregator in `float16` with the depth/camera heads forced to `float32`, instead of `bfloat16` end-to-end. This is an existing, supported code path - not a workaround added here. T4's 16GB VRAM is comfortably above the ~7.4-9.4GB the default window sizing needs, so this is mainly a numerical-precision difference from the paper's reference RTX 5070 Ti/RTX 3050 (both Ampere+) runs, not a capacity problem.

Before running: **Settings -> Accelerator -> GPU T4 x2** (or P100), and **internet: On** (requires one-time phone/SMS verification on your Kaggle account if not already done).

In [ ]:
import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
    print("compute capability:", torch.cuda.get_device_capability(0))
    free_mb, total_mb = [x / 1e6 for x in torch.cuda.mem_get_info()]
    print("VRAM: %.0f MB free / %.0f MB total" % (free_mb, total_mb))
else:
    print("No GPU visible - check Settings > Accelerator is set to GPU T4 x2 or P100.")

## 1. Get the code

Clones the repo (public read access) plus the vendored VGGT submodule. Re-running this cell is safe once cloned.

In [ ]:
import os

REPO_DIR = "/kaggle/working/Terraform"

if not os.path.exists(REPO_DIR):
    !git clone --recurse-submodules https://github.com/Jola7898/Terraform.git {REPO_DIR}
else:
    print("repo already present at", REPO_DIR)

!ls {REPO_DIR}/rtvio/third_party/vggt | head -5

## 2. Install dependencies

Kaggle's GPU image already ships `torch` (with CUDA), `numpy`, `scipy`, and `opencv-python` - only the project's own extras need installing (small, a few seconds each). `pymeshlab`/`trimesh` are for meshing/export; drop them if you only need the raw point cloud.

In [ ]:
!pip install -q -e {REPO_DIR}/rtvio
!pip install -q huggingface_hub einops safetensors pymeshlab trimesh laspy

## 3. VGGT-1B checkpoint

`_load_vggt()` looks for a local checkpoint at `$RTVIO_VGGT_CHECKPOINT` (or `rtvio/data/models/vggt1b_model.pt`), and falls back to downloading `facebook/VGGT-1B` from Hugging Face over the network if neither exists.

- **First run**: leave `CACHED_CKPT` pointing at nothing real - the fallback download happens automatically (~5GB, one-time per session).
- **Subsequent runs**: to avoid re-downloading every session, save the checkpoint once (see the last cell in this notebook), upload the saved `vggt1b_model.pt` as a private Kaggle Dataset, attach it to this notebook (**Add Input**), and point `CACHED_CKPT` at its `/kaggle/input/...` path below.

In [ ]:
import os

# Adjust this to match whatever you name the attached checkpoint dataset.
CACHED_CKPT = "/kaggle/input/vggt1b-checkpoint/vggt1b_model.pt"

if os.path.exists(CACHED_CKPT):
    os.environ["RTVIO_VGGT_CHECKPOINT"] = CACHED_CKPT
    print("using cached checkpoint:", CACHED_CKPT)
else:
    print("no cached checkpoint dataset attached - first reconstruction run below will "
          "download facebook/VGGT-1B from Hugging Face (~5GB, one-time per session)")

## 4. Provide input data

Attach your recording as a Kaggle Dataset (**Add Input**) or drag-and-drop it into the Data pane, then set **one** of:

- `VIDEO_PATH` - a plain video file (`--video`), optionally with `GPS_CSV` (`timestamp_s,lat_deg,lon_deg,alt_m`)
- `SESSION_DIR` - a recorded session directory from `rtvio.studio` or `live_pipeline --record-only` (`--from-recording`)

Leave the one you're not using set to `None`.

In [ ]:
# --- edit these for your data ---
VIDEO_PATH = "/kaggle/input/your-dataset-name/your_clip.mp4"   # or None
GPS_CSV = None                                                  # optional, --video only
SESSION_DIR = None                                              # or a recorded-session path, instead of VIDEO_PATH

OUT_DIR = "/kaggle/working/output"
os.makedirs(OUT_DIR, exist_ok=True)

assert (VIDEO_PATH is not None) != (SESSION_DIR is not None), \
    "set exactly one of VIDEO_PATH or SESSION_DIR"

In [ ]:
import subprocess, sys

cmd = [sys.executable, "-m", "rtvio.vggt_reconstruct", "--out", OUT_DIR, "--extras"]
if VIDEO_PATH:
    cmd += ["--video", VIDEO_PATH]
    if GPS_CSV:
        cmd += ["--gps", GPS_CSV]
else:
    cmd += ["--from-recording", SESSION_DIR]

print("running:", " ".join(cmd))
subprocess.run(cmd, check=True, cwd=os.path.join(REPO_DIR, "rtvio"))

## 5. Collect outputs

Expect (per `rtvio/README.md`'s output-file table): `cloud_raw.ply`, `mesh_poisson.ply`/`.obj`/`.glb`, `cloud.las` (from `--extras`), a DSM raster, `cameras.json`, and `CHECKPOINT_REPORT.md`/`.json` (the per-run validation report: seam residuals, blur, coverage, timing).

Anything under `/kaggle/working` is downloadable from the notebook's Output tab once you **Save Version** - zip it first for a single-file download.

In [ ]:
!ls -la {OUT_DIR}
!cat {OUT_DIR}/CHECKPOINT_REPORT.md 2>/dev/null || echo "(no report found - check the run above for errors)"
!cd /kaggle/working && zip -qr rtvio_output.zip output && ls -la rtvio_output.zip

## 6. Optional, one-time: save the checkpoint for reuse

Only needed once. Run this after a session that downloaded `facebook/VGGT-1B` via the network fallback, to avoid repeating that download on every future session. This writes the same plain `state_dict` format `_load_vggt()` expects at `RTVIO_VGGT_CHECKPOINT` / `data/models/vggt1b_model.pt` - loaded on CPU in float32 so the file is portable regardless of which GPU dtype (bf16/fp16) a given session later casts it to.

After running this cell: **Save Version -> New Dataset** from `/kaggle/working/vggt1b_model.pt`, then attach that dataset as an input to future notebook sessions and point `CACHED_CKPT` (cell above) at it.

In [ ]:
import sys, torch

sys.path.insert(0, os.path.join(REPO_DIR, "rtvio", "src"))
from rtvio.vggt_reconstruct import _load_vggt

model, dtype = _load_vggt("cpu")  # cpu -> stays float32, the portable on-disk format
torch.save(model.state_dict(), "/kaggle/working/vggt1b_model.pt")
print("saved -> /kaggle/working/vggt1b_model.pt (now: Save Version -> New Dataset)")